# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjalanano-dev/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1:** "AI-generated traffic percentages significantly correlate with subsequent CTR drops."
**My Methodology Question:** How exactly is the 'CTR drop' label computed, and what is its time window relative to the AI traffic measurement? If the windows overlap, there is a risk of temporal leakage. Furthermore, does the validation design group by client to ensure the model isn't just memorizing one specific client's traffic characteristics?

**Finding 2:** "Content that hasn't been updated in 90 days has a 60% higher chance of entering a declining trend."
**My Methodology Question:** How do we define the 'declining trend' label, and is it completely independent of the update timestamp? Also, what is the base rate of decline in the dataset? A 60% higher chance is a strong relative claim; seeing the absolute base rate and ensuring the baseline was evaluated on the same honest split would make the claim fully verifiable.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Here, I'll compare a naive random split (Flawed) with a `GroupShuffleSplit` grouped by `client_id` (Honest). Random splits allow the model to learn client-specific behaviors (leakage), inflating the score. The grouped split forces the model to generalize to unseen clients.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

# Load Data
import pathlib
try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = os.getcwd()

if notebook_dir.endswith('notebooks'):
    REPO_ROOT = os.path.abspath(os.path.join(notebook_dir, '..', '..'))
else:
    REPO_ROOT = os.path.abspath(notebook_dir)

DATA_PATH = os.path.join(REPO_ROOT, 'data', 'raw', 'content_refresh_anonymized.csv')
df = pd.read_csv(DATA_PATH)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
df['avg_position'] = df['avg_position'].replace(0, np.nan)

raw_features = ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 
                'days_since_last_update', 'word_count', 'engagement_rate', 
                'scroll_rate', 'ai_traffic_pct']
for col in ['word_count', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']:
    df[f'has_{col}'] = df[col].notna().astype(int)

features = raw_features + [f'has_{col}' for col in ['word_count', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']]
imputer = SimpleImputer(strategy='median')
X_full = pd.DataFrame(imputer.fit_transform(df[features]), columns=features)
y_full = df['is_declining_label']
groups = df['client_id']

def precision_at_k(y_true, scores, k):
    idx = np.argsort(scores)[::-1][:k]
    return np.mean(y_true.iloc[idx])

# 1. Flawed: Random Split
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(X_full, y_full, test_size=0.25, random_state=42)
rf_rand = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf_rand.fit(X_train_rand, y_train_rand)
probs_rand = rf_rand.predict_proba(X_test_rand)[:, 1]

# 2. Honest: Grouped Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X_full, y_full, groups))
X_train_grp, X_test_grp = X_full.iloc[train_idx], X_full.iloc[test_idx]
y_train_grp, y_test_grp = y_full.iloc[train_idx], y_full.iloc[test_idx]
rf_grp = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf_grp.fit(X_train_grp, y_train_grp)
probs_grp = rf_grp.predict_proba(X_test_grp)[:, 1]

# Compare Results
print("Comparison of P@100:")
print(f"Random Split (Flawed): {precision_at_k(y_test_rand, probs_rand, 100):.4f}")
print(f"Grouped Split (Honest): {precision_at_k(y_test_grp, probs_grp, 100):.4f}")
print("\nThe Random Split inflates performance because the model memorizes client baselines.")


Comparison of P@100:
Random Split (Flawed): 0.8800
Grouped Split (Honest): 0.5500

The Random Split inflates performance because the model memorizes client baselines.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
# 1. Check for Label-derived features
# Our label is derived from `trend_direction`. Let's ensure neither `trend_direction` nor `trend_pct` is in our feature set.
print("Features used:")
for f in features:
    if 'trend' in f:
        print(f"WARNING: LEAKAGE DETECTED in feature '{f}'")
print("No trend-based features found.\n")

# 2. Check Feature Importances for suspiciously perfect predictors
importances = pd.DataFrame({'Feature': features, 'Importance': rf_grp.feature_importances_}).sort_values('Importance', ascending=False)
print("Top 5 Feature Importances (Honest Model):")
print(importances.head(5).to_string(index=False))
print("\nObservation: No single feature towers at >50% importance, which usually indicates leakage.")

# 3. Check for Decision-derived features
# We explicitly did NOT include `health_score` or any FlyRank rule-based flags in our features.


Features used:
No trend-based features found.

Top 5 Feature Importances (Honest Model):
         Feature  Importance
 impressions_90d    0.325091
has_avg_position    0.167926
    avg_position    0.097356
  has_word_count    0.093802
      word_count    0.085470

Observation: No single feature towers at >50% importance, which usually indicates leakage.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Flawed, bold claim:** 
"Our model predicts exactly when content will decline and tells you which pages to fix."

**Honest, safe rewrite:** 
"Our model provides decision-support by identifying content with observed signals that directionally align with historical declining trends, helping prioritize pages for review."


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.